# Rewards Analysis Pipeline
- **Pipeline ID:** REWARDS_CALC_002
- **Owner:** JRAJBHUSHAN
- **Team:** Rewards Engineering
- **Description:** Aggregates transaction rewards by account and category, assigns customer reward tiers (Bronze/Silver/Gold/Platinum), and produces a final customer rewards summary table.

*Copyright (c) 2026 Capital One. All rights reserved.*
*SPDX-License-Identifier: UNLICENSED*

In [ ]:
CREATE OR REPLACE TABLE LOB_DB_COLLAB.LOB_REWARDS.REWARD_ANALYSIS_BASE AS
SELECT
    tr.ACCOUNT_ID,
    ct.SPENDING_CATEGORY,
    COUNT(*) AS TXN_COUNT,
    SUM(ct.TRANSACTION_AMOUNT) AS TOTAL_SPEND,
    SUM(tr.POINTS_EARNED) AS TOTAL_POINTS
FROM LOB_DB_COLLAB.LOB_REWARDS.STG_TRANSACTION_REWARDS tr
JOIN LOB_DB_COLLAB.LOB_REWARDS.STG_CATEGORIZED_TRANSACTIONS ct
    ON tr.TRANSACTION_ID = ct.TRANSACTION_ID
GROUP BY 1, 2;

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

df = session.table("LOB_DB_COLLAB.LOB_REWARDS.REWARD_ANALYSIS_BASE").to_pandas()

customer_summary = df.groupby("ACCOUNT_ID").agg(
    TOTAL_SPEND=("TOTAL_SPEND", "sum"),
    TOTAL_POINTS=("TOTAL_POINTS", "sum"),
    CATEGORY_COUNT=("SPENDING_CATEGORY", "nunique"),
    TXN_COUNT=("TXN_COUNT", "sum")
).reset_index()

customer_summary["REWARD_TIER"] = pd.cut(
    customer_summary["TOTAL_POINTS"],
    bins=[0, 500, 2000, 5000, float("inf")],
    labels=["Bronze", "Silver", "Gold", "Platinum"]
)

customer_summary["AVG_POINTS_PER_TXN"] = round(
    customer_summary["TOTAL_POINTS"] / customer_summary["TXN_COUNT"], 2
)

session.write_pandas(
    customer_summary,
    "CUSTOMER_TIER_ANALYSIS",
    database="LOB_DB_COLLAB",
    schema="LOB_REWARDS",
    auto_create_table=True,
    overwrite=True
)

print(customer_summary["REWARD_TIER"].value_counts())

In [ ]:
CREATE OR REPLACE TABLE LOB_DB_COLLAB.LOB_REWARDS.CUSTOMER_REWARDS_FINAL
    COMMENT = 'Final customer rewards summary with tier assignments'
AS
SELECT
    ct.ACCOUNT_ID,
    ct.TOTAL_SPEND,
    ct.TOTAL_POINTS,
    ct.CATEGORY_COUNT,
    ct.TXN_COUNT,
    ct.REWARD_TIER,
    ct.AVG_POINTS_PER_TXN,
    CURRENT_TIMESTAMP() AS PROCESSED_AT
FROM LOB_DB_COLLAB.LOB_REWARDS.CUSTOMER_TIER_ANALYSIS ct;

## Exchange Registration
---
**Table:** LOB_DB_COLLAB.LOB_REWARDS.REWARD_ANALYSIS_BASE
**Description:** Aggregated transaction counts, spend, and points by account and spending category
**Columns:**
- ACCOUNT_ID: Customer account identifier
- SPENDING_CATEGORY: Derived spending category
- TXN_COUNT: Number of transactions
- TOTAL_SPEND: Sum of transaction amounts
- TOTAL_POINTS: Sum of reward points earned

---
**Table:** LOB_DB_COLLAB.LOB_REWARDS.CUSTOMER_TIER_ANALYSIS
**Description:** Customer-level summary with reward tier assignments
**Columns:**
- ACCOUNT_ID: Customer account identifier
- TOTAL_SPEND: Total spend across all categories
- TOTAL_POINTS: Total reward points earned
- CATEGORY_COUNT: Number of distinct spending categories
- TXN_COUNT: Total transaction count
- REWARD_TIER: Assigned tier (Bronze/Silver/Gold/Platinum)
- AVG_POINTS_PER_TXN: Average points earned per transaction

---
**Table:** LOB_DB_COLLAB.LOB_REWARDS.CUSTOMER_REWARDS_FINAL
**Description:** Final customer rewards summary with tier assignments and processing timestamp
**Columns:**
- ACCOUNT_ID: Customer account identifier
- TOTAL_SPEND: Total spend across all categories
- TOTAL_POINTS: Total reward points earned
- CATEGORY_COUNT: Number of distinct spending categories
- TXN_COUNT: Total transaction count
- REWARD_TIER: Assigned tier (Bronze/Silver/Gold/Platinum)
- AVG_POINTS_PER_TXN: Average points earned per transaction
- PROCESSED_AT: Timestamp when the record was created

In [ ]:
SELECT MERCHANT_NAME, SPENDING_CATEGORY, SUM(POINTS_EARNED) AS TOTAL_POINTS
FROM LOB_DB_COLLAB.LOB_REWARDS.STG_TRANSACTION_REWARDS
GROUP BY MERCHANT_NAME, SPENDING_CATEGORY
ORDER BY TOTAL_POINTS DESC
LIMIT 20

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pivot_df = rewards_by_vendor.pivot_table(index='MERCHANT_NAME', columns='SPENDING_CATEGORY', values='TOTAL_POINTS', aggfunc='sum', fill_value=0)
pivot_df = pivot_df.loc[pivot_df.sum(axis=1).sort_values(ascending=True).index]

fig, ax = plt.subplots(figsize=(10, 6))
pivot_df.plot(kind='barh', stacked=True, ax=ax, colormap='Set2')
ax.set_xlabel('Total Points Earned')
ax.set_ylabel('Merchant')
ax.set_title('Rewards Points by Vendor (Stacked by Category)')
ax.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()